In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Install & Load Requirements

In [ ]:
!pip install --upgrade datasets
#!pip install -U fsspec==2024.10.0
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q trl #SFTTrainer
#!pip install -q wandb
!pip install accelerate
!pip install fastparquet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 15.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ...

In [ ]:
import torch
import pandas as pd
import os
import re
from fastparquet import ParquetFile
from transformers import EarlyStoppingCallback
from transformers import TrainingArguments
from datasets import load_dataset, Dataset
from transformers import (AutoModelForCausalLM,
    AutoTokenizer,BitsAndBytesConfig,
    HfArgumentParser,TrainingArguments,
    pipeline,logging)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
import json
import numpy as np
from huggingface_hub import notebook_login
notebook_login()

# Load & Prepare Data

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/SemEval2025Task8Data/final2.csv")
training_data = []
for _, row in df.iterrows():
    prompt = (
        f"Instructions: Generate a single line of Python code that produces the specified output type as Answer type based on the given context. "
        f"Assume the data is already loaded into a DataFrame df. Do not include any additional explanations, imports, variable assignment or text beyond the code and use the exact column names as given in context. If there are columns names like 'author_name<gx:category>' 'user_followers_count<gx:number>' then use it with <gx:number> as it is included in column name. "
        f"Context: Question:{row['question']} Columns to be used:{row['columns_used']} "
        f"Column dtypes:{row['column_types']} Answer type:{row['type']}"
    )
    completion = f"{row['Code Line']}"
    training_data.append({"prompt": prompt, "completion": completion})
training_data[0]

{'prompt': "Instructions: Generate a single line of Python code that produces the specified output type as Answer type based on the given context. Assume the data is already loaded into a DataFrame df. Do not include any additional explanations, imports, variable assignment or text beyond the code and use the exact column names as given in context. If there are columns names like 'author_name<gx:category>' 'user_followers_count<gx:number>' then use it with <gx:number> as it is included in column name. Context: Question:Is the person with the highest net worth self-made? Columns to be used:['finalWorth', 'selfMade'] Column dtypes:['number[uint32]' 'boolean'] Answer type:boolean",
 'completion': 'df[df.finalWorth == max(df.finalWorth)]["selfMade"].values[0]'}

In [ ]:
dataset = Dataset.from_list(training_data)
train_test_split = dataset.train_test_split(test_size=0.1, seed=0)
train_dataset = train_test_split['train']
validation_dataset = train_test_split['test']

In [ ]:
model_id = "meta-llama/CodeLlama-7b-Instruct-hf"
check_point = "/content/drive/MyDrive/SemEval2025Task8Data/Finetunned_CodeLlama/CodeLlamaInstruct_finetuned/checkpoint-91"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

q_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Base Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id ,
                                          torch_dtype="auto")
tokenizer.use_default_system_prompt = False

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
                                         model_id,
                                         low_cpu_mem_usage=True,
                                         quantization_config=q_config,
                                         device_map='auto')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# FineTuning

In [ ]:
# Resuming from previous checkpoint
model = AutoModelForCausalLM.from_pretrained(
                                         check_point,
                                         low_cpu_mem_usage=True,
                                         quantization_config=q_config,
                                         device_map='auto')

## LORA config

In [ ]:
def formatting_prompts_func(x):
    output_texts = []
    for i in range(len(x['prompt'])):
        text = f"{x['prompt'][i]}\n ### Answer: {x['completion'][i]}"
        output_texts.append(text)
    return output_texts

peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
output_dir = "/content/drive/MyDrive/SemEval2025Task8Data/Finetunned_CodeLlama/CodeLlamaInstruct_finetuned"
os.makedirs(output_dir, exist_ok=True)

## Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    learning_rate=1e-4,
    logging_steps=1,
    num_train_epochs=20,
    eval_strategy="epoch",
    save_strategy = "epoch",
    lr_scheduler_type="linear",
    load_best_model_at_end = True,
    metric_for_best_model="eval_loss",
    report_to="tensorboard",
    push_to_hub=True
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
    formatting_func=formatting_prompts_func,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

Map:   0%|          | 0/889 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

# Training

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.906600,1.762843


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.906600,1.762843
2,0.882900,0.801783
3,0.591400,0.577282
4,0.488300,0.496204
5,0.484600,0.477605
6,0.455400,0.465488
7,0.448600,0.457199
8,0.441300,0.453956


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/

TrainOutput(global_step=56, training_loss=0.8412901604814189, metrics={'train_runtime': 3583.6758, 'train_samples_per_second': 1.985, 'train_steps_per_second': 0.016, 'total_flos': 5.04778104386519e+16, 'train_loss': 0.8412901604814189, 'epoch': 8.0})

In [ ]:
# Resume from Checkpoint
trainer.train(resume_from_checkpoint=check_point)

/usr/local/lib/python3.10/dist-packages/transformers/trainer.py:3442: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(checkpoint, OPTIMIZER_NAME), map_

Epoch,Training Loss,Validation Loss
9,0.651900,0.617194
10,0.413700,0.410161
11,0.364500,0.358154
12,0.368200,0.355059
13,0.354600,0.353925


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/

TrainOutput(global_step=91, training_loss=0.18955763748713902, metrics={'train_runtime': 2127.864, 'train_samples_per_second': 5.431, 'train_steps_per_second': 0.043, 'total_flos': 8.400138682284442e+16, 'train_loss': 0.18955763748713902, 'epoch': 13.0})

In [ ]:
# Resume from Checkpoint
trainer.train(resume_from_checkpoint=check_point)

/usr/local/lib/python3.10/dist-packages/transformers/trainer.py:3442: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(checkpoint, OPTIMIZER_NAME), map_

Epoch,Training Loss,Validation Loss
14,0.760300,0.698314
15,0.384000,0.347060
16,0.277500,0.282604
17,0.268600,0.277632
18,0.274400,0.274960


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/

Epoch,Training Loss,Validation Loss
14,0.760300,0.698314
15,0.384000,0.347060
16,0.277500,0.282604
17,0.268600,0.277632
18,0.274400,0.274960
19,0.267600,0.273304
20,0.270300,0.272713


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=140, training_loss=0.14579738953283855, metrics={'train_runtime': 3551.8167, 'train_samples_per_second': 5.006, 'train_steps_per_second': 0.039, 'total_flos': 1.408144896136151e+17, 'train_loss': 0.14579738953283855, 'epoch': 20.0})

In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/asimali004/CodeLlamaInstruct_finetuned/commit/6743f30a801d470072877eb068fb97abb14e0214', commit_message='End of training', commit_description='', oid='6743f30a801d470072877eb068fb97abb14e0214', pr_url=None, repo_url=RepoUrl('https://huggingface.co/asimali004/CodeLlamaInstruct_finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='asimali004/CodeLlamaInstruct_finetuned'), pr_revision=None, pr_num=None)

## Saving final model

In [ ]:
model_to_save = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model  # Take care of distributed/parallel training
model_to_save.save_pretrained(output_dir)

## Tensorboard Plots

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/SemEval2025Task8Data/Finetunned_CodeLlama/CodeLlamaInstruct_finetuned/runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

# Inference

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id ,
                                          torch_dtype="auto")
tokenizer.use_default_system_prompt = False

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model = AutoModelForCausalLM.from_pretrained(
                                         check_point,
                                         low_cpu_mem_usage=True,
                                         quantization_config=q_config,
                                         device_map='auto')


In [ ]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32016, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/SemEval2025Task8Data/Data_complete/test.csv")

In [ ]:
testing_data = []
for _, row in df.iterrows():
    prompt = (
    f"Instructions: Generate a single line of Python code that produces the specified output type as the Answer type, based strictly on the given context. "
    f"Assume the data is already loaded into a DataFrame df. Do not include any additional explanations, imports, variable assignments, or text beyond the code. "
    f"Use the exact column names as provided in the context, including any special characters like '<gx:category>' or '<gx:number>'. "
    f"Context: Question: {row['question']} Columns to be used: {row['columns_used']} "
    f"Column dtypes: {row['column_types']} Answer type: {row['type']}"
    )
    testing_data.append({"prompt": prompt})
testing_data[0]

{'prompt': "Instructions: Generate a single line of Python code that produces the specified output type as the Answer type, based strictly on the given context. Assume the data is already loaded into a DataFrame df. Do not include any additional explanations, imports, variable assignments, or text beyond the code. Use the exact column names as provided in the context, including any special characters like '<gx:category>' or '<gx:number>'. Context: Question: Is the most favorited author mainly communicating in Spanish? Columns to be used: ['favorites' 'lang'] Column dtypes: ['category' 'category'] Answer type: boolean"}

In [ ]:
dataset = Dataset.from_list(testing_data)
dataset

Dataset({
    features: ['prompt'],
    num_rows: 320
})

In [ ]:
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=320, device_map="auto", batch_size=32)

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausalLM', 'GraniteMoeForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'Llama

In [ ]:
prompt = dataset[96]["prompt"]
li = f"<s>[INST] {prompt} [/INST]"
result = pipe(li)
print(result[0]['generated_text'][len(li):])

  df.groupby("author_name<gx:category>")["user_followers_count<gx:number>"].nlargest(3).index.tolist()


In [ ]:
from torch.utils.data import DataLoader

batch_size = 32
outputs = []

prompts = [f"<s>[INST] {a['prompt']} [/INST]" for a in dataset]
data_loader = DataLoader(prompts, batch_size=batch_size)

counter = 0
for batch in data_loader:
    results = pipe(batch)

    for idx, result in enumerate(results):
        generated_text = result[0]['generated_text'][len(batch[idx]):]
        outputs.append(generated_text)


    counter += 1
    print(f"Processed batch {counter}")

Processed batch 1
Processed batch 2
Processed batch 3
Processed batch 4
Processed batch 5
Processed batch 6
Processed batch 7
Processed batch 8


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed batch 9
Processed batch 10


In [ ]:
df["Instruct_lines"] = outputs

In [ ]:
df.to_csv('/content/drive/MyDrive/SemEval2025Task8Data/Data_complete/Instruct_lines1.csv', index=False)

# Post-Processing And Code Line Execution

In [ ]:
csv_file = "/content/drive/MyDrive/SemEval2025Task8Data/Data_complete/Instruct_lines1.csv"
parquet_folder = "/content/drive/MyDrive/SemEval2025Task8Data/data"
df2 = pd.read_csv(csv_file)

df2["Instruct_answer"] = None
df2["Error"] = None
df2["Evaluated_code"] = None
c_dataset = None
variables = {}

for index, row in df2.iterrows():
    dataset_name = row["dataset"]
    code_line = row["Instruct_lines"]

    parquet_file = os.path.join(parquet_folder, dataset_name, "all.parquet")
    if not os.path.exists(parquet_file):
        print(f"Parquet file not found for index {index}: {parquet_file}")
        df2.at[index, "Instruct_answer"] = "Parquet file not found"
        df2.at[index, "Error"] = "Parquet file not found"
        continue

    if dataset_name != c_dataset:
        try:
            table = ParquetFile(parquet_file)
            df = table.to_pandas()
            c_dataset = dataset_name
        except Exception as e:
            print(f"Error reading Parquet file for index {index}: {e}")
            df2.at[index, "Error"] = f"Error reading Parquet: {e}"
            continue
    try:
        # Remove Un-necessary Text from generated code lines if any
        try:
            code_line = re.sub(r'`', '', code_line)
            code_line = re.sub(r'```', '', code_line)
            code_line = re.sub(r'```\n', '', code_line)
            code_line = re.sub(r'<sos>|<eos>', '', code_line)
            code_line = code_line.strip()
        except Exception as e:
            print(f"Error cleaning code_line at index {index}: {e}")
            df2.at[index, "Error"] = f"Backtick cleanup error: {e}"
            continue

        # Extract code line if any variable assignment found
        if re.match(r"^\s*[a-zA-Z_]\w*\s*=", code_line):
            variable, expression = code_line.split('=', 1)
            variable = variable.strip()
            expression = expression.strip()

            # Evaluate the Code Line
            result = eval(expression, {"df": df}, variables)
            variables[variable] = result
            df2.at[index, "Instruct_answer"] = f"{variable} = {result}"
            df2.at[index, "Evaluated_code"] = f"{variable} = {expression}"
        else:
            # Evaluate the Code Line
            result = eval(code_line, {"df": df}, variables)
            df2.at[index, "Instruct_answer"] = result
            df2.at[index, "Evaluated_code"] = code_line

    except Exception as e:
        print(f"Error executing code for index {index}: {e}")
        df2.at[index, "Error"] = f"Execution error: {e}"

# Remove unnamed column if found
df2 = df2.loc[:, ~df2.columns.str.contains('^Unnamed')]
error_rows = df2[df2["Error"].notna()]
output_file = "output_file.csv"
error_file = "error_rows.csv"
df2.to_csv(output_file, index=False)
error_rows.to_csv(error_file, index=False)

print("Processing complete.")


Error executing code for index 1: attempt to get argmax of an empty sequence
Error executing code for index 19: Column 'text' has dtype object, cannot use method 'nlargest' with this dtype
Error executing code for index 21: invalid decimal literal (<string>, line 1)
Error executing code for index 41: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


<string>:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


Error executing code for index 78: 'num_claim'
Error executing code for index 80: 'user_verified'
Error executing code for index 81: unmatched ']' (<string>, line 1)
Error executing code for index 87: Series.count() takes 1 positional argument but 2 were given
Error executing code for index 94: unhashable type: 'list'
Error executing code for index 96: 'user_followers_count'
Error executing code for index 98: unhashable type: 'list'
Error executing code for index 100: 'numpy.uint8' object has no attribute 'eq'
Error executing code for index 103: 'numpy.uint16' object has no attribute 'eq'
Error executing code for index 115: 391


<string>:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


Error executing code for index 158: 'User self-placement on Progressive-Conservative economic values axis'
Error executing code for index 160: unsupported operand type(s) for +: 'int' and 'str'
Error executing code for index 166: 'What is the highest degree or level of school you have *completed*?'
Error executing code for index 170: 'What is the highest degree or level of school you have *completed*?'
Error executing code for index 176: 'int' object has no attribute 'nlargest'


<string>:1: FutureWarning: 'any' with datetime64 dtypes is deprecated and will raise in a future version. Use (obj != pd.Timestamp(0)).any() instead.


Error executing code for index 229: 'int' object is not subscriptable
Error executing code for index 242: invalid decimal literal (<string>, line 1)
Error executing code for index 243: unmatched ']' (<string>, line 1)
Error executing code for index 263: invalid decimal literal (<string>, line 1)
Processing complete.


## Accuracy

In [ ]:
def calculate_accuracy(df):
    df['Instruct_answer'] = df['Instruct_answer'].astype(str)
    df['answer'] = df['answer'].astype(str)
    correct_predictions = (df['Instruct_answer'] == df['answer']).sum()
    total_predictions = len(df)
    if total_predictions == 0:
        return 0.0, df, pd.DataFrame()
    accuracy = correct_predictions / total_predictions
    mismatched_rows = df[df['Instruct_answer'] != df['answer']]
    return accuracy, mismatched_rows

accuracy, mismatched_rows = calculate_accuracy(df2)
print(f"Accuracy: {accuracy:.2%}")
mismatched_rows.to_csv("mismatched_rows.csv", index=False)


Accuracy: 66.25%
